# 23 · Apply ruleset, checked against an independent decision table

## What this checks

**Apply ruleset** reaches a classification by rules. For example, it flags a location for review when an alert is
active and the distance exceeds a threshold. This check covers Apply ruleset release 0.1.0, and three rulesets it
runs.

Each of those rulesets was once a widget of its own: **Outreach criteria**, **Access review** and **Diagnostic
evidence**. All three widgets are retired. Apply ruleset hosted the three rulesets as data, until they were retired
too, with the Old Naledi example, on 11 November 2026. The check covers the releases those widgets were retired at,
and Apply ruleset as their host. It is the same check, because they are the same rules. The rule text was proved
identical, byte for byte, when the rulesets became data.

So this notebook is now the record of rulesets Fieldwork no longer carries.

Fieldwork writes the facts and the rules in N3, a language for stating facts and rules. It hands both to EYE, a
reasoner: a program that draws conclusions from rules.

There are **two claims** here, and they fail in different ways:

1. **The logic gives the same result.** The conclusions the reasoner reached match the same logic written plainly in
   another language.
2. **The receipt is the same receipt.** The record shown to a reader holds what the reasoner was actually given.

A right answer nobody can audit is worth little. So is a tidy receipt for the wrong conclusion. Both are needed.

## How it is checked

**For the logic**, we write a decision table in Python for each ruleset. A decision table lists every combination
of inputs and the conclusion each should give. It is written from the ruleset's own description in words, not copied
from its N3. For outreach criteria, the description says: "flags a location for review when an alert is active and
its distance exceeds a threshold; missing evidence is concluded unknown rather than negative". That becomes three
lines of Python. If that reading and the shipped rules disagree, the disagreement is the finding.

There is no rule engine and no N3 on this side, and no code is shared with Fieldwork.

**For the receipt**, we read the facts and rules the run emitted, with a small reader written here for the plain kind
of N3 the receipts use. We use no RDF library. We did not add a package to the browser kernel, which this lab has
measured crashing on some package combinations. The cost is that this reader is itself the judge. So it first checks
that it can read its own output back.

**Every possible input is checked, not a sample.** Each of the three rulesets has a fixed number of possible inputs:
12 for outreach criteria, 10 for access review and 24 for the evidence ruleset. That is 46 in all, and every one is
checked. A sample cannot see three things this can:

- a combination no rule fires on;
- a combination where two rules disagree;
- the boundary case, where a value **equals** the threshold.

## What should happen

For every one of the 46 possible inputs:

1. **The conclusion** from the reasoner is the one the decision table gives.
2. **Only declared facts.** The receipt states only the facts the ruleset says it takes, and no others.
3. **Every number** carries the declared nine decimal places.
4. **One subject.** Each case concerns one subject.
5. **No named place.** No rule names a particular place or record. A ruleset classifies; it must not list cases.
6. **No blank left.** No setting in a rule is left unfilled.
7. **Only comparisons.** The rules use only simple comparisons of numbers.

These criteria were fixed before the check was run.

In [ ]:
# Nothing to load: a decision table is comparisons on values the fixture already carries,
# and the N3 reader is forty lines in the check module. Adding a parser to this kernel was
# declined deliberately — 00-kernel-check records what package combinations kill it.
print("Nothing to load.")


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_23_rulesets import run

report = run(Path.cwd())
print("environment      :", report["environment"])
print("logic agrees     :", report["logicAgrees"])
print("receipts agree   :", report["receiptsAgree"])
print("cells checked    :", report["totalCells"], "— the complete input space of all three rulesets")
print("agrees           :", report["agrees"])


## The result on record

Recorded on 12 October 2026, against release 0.1.0: **Agrees**. It was not rerun on 26 November 2026: the three
rulesets are retired, so nothing can export their data again.

All 46 possible inputs agree, on the conclusion and on the receipt:

- **Coverage.** No input was left with no rule firing. No input had two conclusions for the same question.
- **The rules.** The three rulesets hold 19 rules in all.
- **The comparisons.** They use exactly two: "greater than" and "not greater than". There is no negation, no rule
  that calls itself and no totalling.
- **The boundary.** The case where a value equals the threshold is in every table.

Running the cell above repeats the check on the same data.

## The decision tables, every cell

Green means the reasoner agrees with an independent reading of the description.

Look hardest at the row in each table where the measured value **equals** the threshold. "Not greater than" puts
that value on the safe side. "Less than" would not. That one row is what tells the two apart, and a sample would
likely miss it.

In [ ]:
from IPython.display import HTML, display

def table(entry):
    rows = entry['rows']
    keys = [k for k in rows[0]['inputs'] if k not in ('subjectId',)]
    head = ''.join(f'<th style="text-align:left;padding:2px 8px">{k}</th>' for k in keys)
    body = ''
    for row in rows:
        tint = '#e6f4ea' if row['agrees'] else '#fce8e6'
        mark = '✓' if row['agrees'] else '✗'
        cells = ''.join(f'<td style="padding:2px 8px">{row["inputs"][k]}</td>' for k in keys)
        emitted = ', '.join(f'{p.split(":")[1]}={v.split(":")[1]}' for p, v in sorted(row['emitted'].items()))
        expected = ', '.join(f'{p.split(":")[1]}={v.split(":")[1]}' for p, v in sorted(row['expected'].items()))
        shown = emitted if row['agrees'] else f'reasoner: {emitted}<br>python: {expected}'
        body += f'<tr style="background:{tint}">{cells}<td style="padding:2px 8px">{shown}</td>'
        body += f'<td style="padding:2px 8px;text-align:center">{mark}</td></tr>'
    return (f'<h4 style="margin:12px 0 4px">{entry["ruleset"].split(":")[-1]} '
            f'<span style="font-weight:normal;color:#666">{entry["rulesetVersion"]} · '
            f'{entry["cells"]} cells · basis {entry["basisStatus"]}</span></h4>'
            f'<table style="border-collapse:collapse;font:12px/1.5 ui-monospace,monospace">'
            f'<tr>{head}<th style="text-align:left;padding:2px 8px">conclusion</th>'
            f'<th style="padding:2px 8px"></th></tr>{body}</table>')

for entry in report['rulesets']:
    display(HTML(table(entry)))


## The receipt, checked rather than trusted

A conclusion a reader cannot audit is worth little. These are the facts and rules the run actually emitted, not a
description of them.

In the output below:

- **Undeclared** counts facts in the receipt that the ruleset does not declare it takes.
- **Instances in rules** matters most. A ruleset classifies, and must not list cases.

The `nb06-evidence` ruleset records a real defect: two facilities are matched by their **name as text**. The check
shows that this match happens where the facts are made, not in the rules.

In [ ]:
for entry in report['rulesets']:
    receipt = entry['receipt']
    print(f"{entry['ruleset'].split(':')[-1]:<20} rules {receipt['ruleCount']:>2}  "
          f"builtins {','.join(b.split(':')[1] for b in receipt['builtins']) or '—':<32} "
          f"undeclared {len(receipt['undeclaredPredicates'])}  "
          f"precision {len(receipt['precisionProblems'])}  "
          f"instances in rules {len(receipt['instancesNamedInRules'])}")
    print(f"{'':<20} declared predicates: {', '.join(p.split(':')[1] for p in receipt['declaredPredicates'])}")
    if receipt['boundValues']:
        print(f"{'':<20} bound into the rules: {', '.join(receipt['boundValues'])}")
    print()


## What this does not show

- That any threshold is the right threshold.
- That any of these rulesets is a good one. The check shows that the rules do what the ruleset says they do. It shows
  that a reader sees what the reasoner was given.
- That the rulesets rest on evidence. One of the three claims a **published** basis. **None** of the three carries
  evidence a reader can open. No check of this kind can fix that. The cell below lists each ruleset's basis.

In [ ]:
for entry in report['rulesets']:
    print(f"{entry['ruleset'].split(':')[-1]:<20} basis {entry['basisStatus']:<13} "
          f"resolvable evidence {entry['resolvableEvidence']}  "
          f"named but unresolvable: {', '.join(entry['unresolvedCitations']) or '—'}")
print()
print('rulesets with evidence a reader can open:', report['rulesetsWithResolvableEvidence'], 'of', len(report['rulesets']))
